# Dimensionnement annuel - periode critique 15 jours

Objectifs du notebook:

- lire le nouveau CSV `chiffresAnnes.csv`;
- nettoyer les colonnes et les nombres avec virgule decimale;
- trouver la pire periode de 15 jours consecutifs pour 250, 290 et 350 panneaux;
- dimensionner le meilleur couple `panneaux / batteries` sur la periode critique.

## 1. Parametres

Les valeurs de cout et de batterie sont regroupees ici pour pouvoir les modifier facilement.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# Le notebook est dans dimentionnement_anne, mais ce fallback permet aussi
# de l'executer depuis la racine du projet.
DATA_CANDIDATES = [
    Path("chiffresAnnes.csv"),
    Path("dimentionnement_anne") / "chiffresAnnes.csv",
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Impossible de trouver chiffresAnnes.csv")

WINDOW_DAYS = 15
PANELS_TO_CHECK = [250, 290, 350]

# Periode de reference pour lancer ensuite l'optimisation.
# A changer si on veut dimensionner sur la pire periode trouvee avec 250 ou 350 panneaux.
PANELS_REFERENCE_PERIOD = 290

# Plage de recherche pour le dimensionnement optimal.
PANELS_MIN = 0
PANELS_MAX = 2000
PANELS_STEP = 1

# Cout des composants.
PANEL_COST_EUR = 139.66
BATTERY_COST_EUR = 449.99

# Batterie: 2.4 kWh nominal, utilisable entre 20% et 90%.
BATTERY_NOMINAL_KWH = 2.4
SOC_MIN = 0.20
SOC_MAX = 0.90
BATTERY_USABLE_KWH = BATTERY_NOMINAL_KWH * (SOC_MAX - SOC_MIN)

print(f"CSV utilise: {DATA_PATH}")
print(f"Capacite utilisable par batterie: {BATTERY_USABLE_KWH:.2f} kWh")

## 2. Lecture et comprehension du CSV

Le fichier utilise `;` comme separateur et des virgules comme separateur decimal. Les noms de colonnes sont d'abord affiches pour verifier la nouvelle structure.

In [ ]:
raw = pd.read_csv(DATA_PATH, sep=";", dtype=str)
raw.columns = raw.columns.str.strip()

print(f"Nombre de lignes: {len(raw)}")
print("Colonnes:")
for col in raw.columns:
    print(f"- {col}")

display(raw.head())

## 3. Nettoyage des donnees

`Puissance PV kW` est interpretee comme la puissance horaire d'un seul panneau, car elle correspond au fichier solaire par panneau (`puissance_panneau_kW`). Les puissances horaires en kW sont ensuite sommees comme des kWh, puisque le pas de temps est horaire.

In [ ]:
def nombre_fr(series):
    """Convertit des nombres francais stockes en texte vers float."""
    return pd.to_numeric(
        series.astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


column_map = {
    "date_heure": "date_heure",
    "Puissance Consume (kW)": "consommation_kW",
    "Vitesse du Vent (ms)": "vitesse_vent_m_s",
    "power_kW_calculated Eolienne2": "eolien_kW",
    "irradiance direct": "irradiance_direct",
    "irradiance difuse": "irradiance_diffuse",
    "irradiance total": "irradiance_total",
    "Puissance PV kW": "pv_un_panneau_kW",
}

df = raw.rename(columns={old: new for old, new in column_map.items() if old in raw.columns}).copy()

required_columns = ["date_heure", "consommation_kW", "eolien_kW", "pv_un_panneau_kW"]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Colonnes manquantes dans le CSV: {missing}")

numeric_columns = [
    "consommation_kW",
    "vitesse_vent_m_s",
    "eolien_kW",
    "irradiance_direct",
    "irradiance_diffuse",
    "irradiance_total",
    "pv_un_panneau_kW",
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = nombre_fr(df[col])

df["date_heure"] = pd.to_datetime(df["date_heure"], dayfirst=True, errors="coerce")

# Les heures du fichier sont etiquetees comme fin d'intervalle: 01:00 represente 00:00-01:00.
# On decale donc d'une heure pour regrouper correctement les 365 jours de 2019.
df["timestamp"] = df["date_heure"] - pd.Timedelta(hours=1)

df = df.sort_values("timestamp").reset_index(drop=True)

print("Valeurs manquantes apres nettoyage:")
display(df[["date_heure", "timestamp", "consommation_kW", "eolien_kW", "pv_un_panneau_kW"]].isna().sum().to_frame("na_count"))

display(df[["date_heure", "timestamp", "consommation_kW", "eolien_kW", "pv_un_panneau_kW"]].head())
display(df[["date_heure", "timestamp", "consommation_kW", "eolien_kW", "pv_un_panneau_kW"]].tail())

## 4. Fonctions de production et bilan

Pour un nombre de panneaux donne:

`production = eolien + panneaux * production_un_panneau`

`bilan = production - consommation`

In [ ]:
def profil_horaire(df_base, panels):
    profil = df_base[["timestamp", "date_heure", "consommation_kW", "eolien_kW", "pv_un_panneau_kW"]].copy()
    profil["panneaux"] = int(panels)
    profil["conso_kWh"] = profil["consommation_kW"]
    profil["eolien_kWh"] = profil["eolien_kW"]
    profil["pv_kWh"] = profil["pv_un_panneau_kW"] * panels
    profil["production_kWh"] = profil["eolien_kWh"] + profil["pv_kWh"]
    profil["bilan_kWh"] = profil["production_kWh"] - profil["conso_kWh"]
    return profil


def resume_annuel(panel_counts):
    lignes = []
    for panels in panel_counts:
        profil = profil_horaire(df, panels)
        lignes.append(
            {
                "panneaux": panels,
                "conso_annuelle_kWh": profil["conso_kWh"].sum(),
                "eolien_annuel_kWh": profil["eolien_kWh"].sum(),
                "pv_annuel_kWh": profil["pv_kWh"].sum(),
                "production_annuelle_kWh": profil["production_kWh"].sum(),
                "bilan_annuel_kWh": profil["bilan_kWh"].sum(),
            }
        )
    return pd.DataFrame(lignes)


display(resume_annuel(PANELS_TO_CHECK).round(2))

## 5. Recherche des pires periodes de 15 jours

La periode critique est calculee avec une fenetre glissante horaire de 15 jours, soit `15 * 24 = 360` heures. On ne fait pas d'agregation journaliere avant le calcul. Le tri principal utilise le bilan 15 jours le plus faible, car il tient compte a la fois de la production et de la consommation.

In [ ]:
def fenetres_15j(df_base, panel_counts, window_days=15):
    resultats = []
    window_hours = window_days * 24

    for panels in panel_counts:
        profil = profil_horaire(df_base, panels).sort_values("timestamp").reset_index(drop=True)
        profil_indexe = profil.set_index("timestamp")

        pas_horaire = profil_indexe.index.to_series().diff().dropna()
        if not pas_horaire.empty and not (pas_horaire == pd.Timedelta(hours=1)).all():
            print(f"Attention: pas horaire non regulier detecte pour {panels} panneaux.")
            display(pas_horaire.value_counts().to_frame("nombre"))

        rolling = profil_indexe[["conso_kWh", "eolien_kWh", "pv_kWh", "production_kWh", "bilan_kWh"]].rolling(
            window_hours,
            min_periods=window_hours,
        ).sum().dropna()
        tmp = rolling.reset_index().rename(
            columns={
                "timestamp": "heure_fin_incluse",
                "conso_kWh": "conso_15j_kWh",
                "eolien_kWh": "eolien_15j_kWh",
                "pv_kWh": "pv_15j_kWh",
                "production_kWh": "production_15j_kWh",
                "bilan_kWh": "bilan_15j_kWh",
            }
        )
        tmp["heure_debut"] = tmp["heure_fin_incluse"] - pd.Timedelta(hours=window_hours - 1)
        tmp["heure_fin_exclue"] = tmp["heure_debut"] + pd.Timedelta(days=window_days)
        tmp["panneaux"] = int(panels)
        resultats.append(tmp)

    resultats = pd.concat(resultats, ignore_index=True)
    return resultats[
        [
            "panneaux",
            "heure_debut",
            "heure_fin_incluse",
            "heure_fin_exclue",
            "conso_15j_kWh",
            "eolien_15j_kWh",
            "pv_15j_kWh",
            "production_15j_kWh",
            "bilan_15j_kWh",
        ]
    ]


fenetres = fenetres_15j(df, PANELS_TO_CHECK, WINDOW_DAYS)

pires_bilan = (
    fenetres.sort_values(["panneaux", "bilan_15j_kWh"])
    .groupby("panneaux", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

pires_production = (
    fenetres.sort_values(["panneaux", "production_15j_kWh"])
    .groupby("panneaux", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

print("Top 5 des pires periodes par bilan production - consommation:")
display(pires_bilan.round(2))

print("Top 5 des plus faibles productions 15 jours:")
display(pires_production.round(2))

In [ ]:
pire_par_panneaux = (
    fenetres.sort_values(["panneaux", "bilan_15j_kWh"])
    .groupby("panneaux", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(pire_par_panneaux.round(2))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(pire_par_panneaux["panneaux"].astype(str), pire_par_panneaux["bilan_15j_kWh"])
ax.set_title("Pire bilan sur 15 jours par nombre de panneaux")
ax.set_xlabel("Nombre de panneaux")
ax.set_ylabel("Bilan 15 jours (kWh)")
ax.axhline(0, color="black", linewidth=0.8)
plt.show()

## 6. Choix de la periode critique pour le dimensionnement

Par defaut, on prend la pire periode trouvee avec 290 panneaux. Modifier `PANELS_REFERENCE_PERIOD` dans les parametres pour refaire le dimensionnement avec 250 ou 350 panneaux comme reference.

In [ ]:
periode_critique = pire_par_panneaux[pire_par_panneaux["panneaux"] == PANELS_REFERENCE_PERIOD]
if periode_critique.empty:
    raise ValueError(f"PANELS_REFERENCE_PERIOD={PANELS_REFERENCE_PERIOD} absent de {PANELS_TO_CHECK}")

periode_critique = periode_critique.iloc[0]
debut = pd.Timestamp(periode_critique["heure_debut"])
fin_inclus = pd.Timestamp(periode_critique["heure_fin_incluse"])
fin_exclus = pd.Timestamp(periode_critique["heure_fin_exclue"])

df_critique = df[(df["timestamp"] >= debut) & (df["timestamp"] < fin_exclus)].copy()

print(f"Periode critique retenue ({PANELS_REFERENCE_PERIOD} panneaux): {debut} -> {fin_exclus} exclu")
print(f"Derniere heure incluse: {fin_inclus}")
print(f"Nombre d'heures dans la periode: {len(df_critique)}")
display(periode_critique.to_frame().T.round(2))

## 7. Dimensionnement optimal panneaux / batteries

Pour chaque nombre de panneaux teste, on calcule le bilan horaire sur la periode critique, puis le besoin minimal en capacite batterie utilisable. La batterie est supposee pleine au debut de la periode critique (`SOC_MAX`).

In [ ]:
def besoin_batterie_utilisable_kWh(bilan_horaire):
    """Besoin utilisable avec batterie pleine au debut et surplus ecrete si elle est deja pleine."""
    bilan = np.asarray(bilan_horaire, dtype=float)
    cumul = np.insert(np.cumsum(bilan), 0, 0.0)
    drawdown = np.maximum.accumulate(cumul) - cumul
    return float(drawdown.max())


def dimensionnement_sur_periode(df_periode, panels_min=0, panels_max=2000, step=1):
    lignes = []
    eolien = df_periode["eolien_kW"].to_numpy(dtype=float)
    pv_un_panneau = df_periode["pv_un_panneau_kW"].to_numpy(dtype=float)
    conso = df_periode["consommation_kW"].to_numpy(dtype=float)

    for panels in range(panels_min, panels_max + 1, step):
        pv = pv_un_panneau * panels
        production = eolien + pv
        bilan = production - conso

        besoin_utilisable = besoin_batterie_utilisable_kWh(bilan)
        batteries = math.ceil(besoin_utilisable / BATTERY_USABLE_KWH) if besoin_utilisable > 0 else 0
        capacite_nominale = batteries * BATTERY_NOMINAL_KWH
        capacite_utilisable = batteries * BATTERY_USABLE_KWH
        cout_panneaux = panels * PANEL_COST_EUR
        cout_batteries = batteries * BATTERY_COST_EUR

        lignes.append(
            {
                "panneaux": panels,
                "bilan_periode_kWh": bilan.sum(),
                "besoin_batterie_utilisable_kWh": besoin_utilisable,
                "batteries": batteries,
                "capacite_nominale_batteries_kWh": capacite_nominale,
                "capacite_utilisable_batteries_kWh": capacite_utilisable,
                "cout_panneaux_EUR": cout_panneaux,
                "cout_batteries_EUR": cout_batteries,
                "cout_total_EUR": cout_panneaux + cout_batteries,
            }
        )

    return pd.DataFrame(lignes)


resultats_dimensionnement = dimensionnement_sur_periode(
    df_critique,
    panels_min=PANELS_MIN,
    panels_max=PANELS_MAX,
    step=PANELS_STEP,
)

top_dimensionnement = resultats_dimensionnement.sort_values("cout_total_EUR").head(10)
best = top_dimensionnement.iloc[0]

display(top_dimensionnement.round(2))

if int(best["panneaux"]) == PANELS_MAX:
    print("Attention: le meilleur resultat est sur PANELS_MAX. Augmenter PANELS_MAX et relancer.")

print(
    f"Meilleur couple: {int(best['panneaux'])} panneaux / {int(best['batteries'])} batteries "
    f"pour un cout total de {best['cout_total_EUR']:.2f} EUR"
)

## 8. Verification par simulation horaire

Cette simulation verifie le couple retenu avec les limites de batterie: 20% minimum et 90% maximum.

In [ ]:
def simuler_batterie(df_base, panels, batteries):
    panels = int(panels)
    batteries = int(batteries)

    capacite_nominale = batteries * BATTERY_NOMINAL_KWH
    energie_min = capacite_nominale * SOC_MIN
    energie_max = capacite_nominale * SOC_MAX
    energie = energie_max

    lignes = []
    for row in df_base.sort_values("timestamp").itertuples(index=False):
        pv = row.pv_un_panneau_kW * panels
        production = row.eolien_kW + pv
        conso = row.consommation_kW
        bilan = production - conso

        energie_non_servie = 0.0
        energie_ecretee = 0.0

        if batteries == 0:
            energie_non_servie = max(0.0, -bilan)
            energie_ecretee = max(0.0, bilan)
            soc_pct = np.nan
        elif bilan >= 0:
            energie_apres = energie + bilan
            energie_ecretee = max(0.0, energie_apres - energie_max)
            energie = min(energie_apres, energie_max)
            soc_pct = 100 * energie / capacite_nominale
        else:
            besoin = -bilan
            disponible = max(0.0, energie - energie_min)
            decharge = min(disponible, besoin)
            energie -= decharge
            energie_non_servie = besoin - decharge
            soc_pct = 100 * energie / capacite_nominale

        lignes.append(
            {
                "timestamp": row.timestamp,
                "date_heure": row.date_heure,
                "panneaux": panels,
                "batteries": batteries,
                "consommation_kWh": conso,
                "eolien_kWh": row.eolien_kW,
                "pv_kWh": pv,
                "production_kWh": production,
                "bilan_kWh": bilan,
                "energie_stockee_kWh": energie,
                "soc_pct": soc_pct,
                "energie_non_servie_kWh": energie_non_servie,
                "energie_ecretee_kWh": energie_ecretee,
            }
        )

    return pd.DataFrame(lignes)


def resume_simulation(simulation):
    return pd.Series(
        {
            "conso_kWh": simulation["consommation_kWh"].sum(),
            "production_kWh": simulation["production_kWh"].sum(),
            "bilan_kWh": simulation["bilan_kWh"].sum(),
            "energie_non_servie_kWh": simulation["energie_non_servie_kWh"].sum(),
            "energie_ecretee_kWh": simulation["energie_ecretee_kWh"].sum(),
            "soc_min_pct": simulation["soc_pct"].min(),
            "soc_max_pct": simulation["soc_pct"].max(),
        }
    )


best_panels = int(best["panneaux"])
best_batteries = int(best["batteries"])

simulation_critique = simuler_batterie(df_critique, best_panels, best_batteries)
display(resume_simulation(simulation_critique).round(2).to_frame("periode_critique"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(simulation_critique["timestamp"], simulation_critique["consommation_kWh"], label="Consommation", color="tab:red")
axes[0].plot(simulation_critique["timestamp"], simulation_critique["production_kWh"], label="Production", color="tab:green")
axes[0].set_ylabel("kWh par heure")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(simulation_critique["timestamp"], simulation_critique["soc_pct"], label="SOC batterie", color="tab:blue")
axes[1].axhline(SOC_MIN * 100, color="tab:red", linestyle="--", linewidth=1, label="SOC min")
axes[1].axhline(SOC_MAX * 100, color="tab:green", linestyle="--", linewidth=1, label="SOC max")
axes[1].set_ylabel("SOC (%)")
axes[1].set_xlabel("Date")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle(f"Simulation periode critique - {best_panels} panneaux / {best_batteries} batteries")
plt.tight_layout()
plt.show()

## 9. Verification annuelle du couple optimal

La cellule suivante applique le couple optimal a toute l'annee. Cela permet de verifier si le dimensionnement obtenu sur la periode critique couvre aussi le reste des donnees.

In [ ]:
simulation_annuelle = simuler_batterie(df, best_panels, best_batteries)
display(resume_simulation(simulation_annuelle).round(2).to_frame("annee_complete"))

## 10. Dimensionnement optimal sur toute l'annee

Cette section calcule aussi le meilleur couple directement sur toute l'annee. Elle est utile si la verification annuelle du couple dimensionne sur la periode critique montre encore de l'energie non servie.

In [ ]:
resultats_dimensionnement_annuel = dimensionnement_sur_periode(
    df,
    panels_min=PANELS_MIN,
    panels_max=PANELS_MAX,
    step=PANELS_STEP,
)

top_dimensionnement_annuel = resultats_dimensionnement_annuel.sort_values("cout_total_EUR").head(10)
best_annuel = top_dimensionnement_annuel.iloc[0]

display(top_dimensionnement_annuel.round(2))

best_annuel_panels = int(best_annuel["panneaux"])
best_annuel_batteries = int(best_annuel["batteries"])
simulation_best_annuel = simuler_batterie(df, best_annuel_panels, best_annuel_batteries)

print(
    f"Meilleur couple annuel: {best_annuel_panels} panneaux / {best_annuel_batteries} batteries "
    f"pour un cout total de {best_annuel['cout_total_EUR']:.2f} EUR"
)
display(resume_simulation(simulation_best_annuel).round(2).to_frame("optimum_annuel"))

if int(best_annuel["panneaux"]) == PANELS_MAX:
    print("Attention: le meilleur resultat annuel est sur PANELS_MAX. Augmenter PANELS_MAX et relancer.")

## 11. Export des resultats

Les exports sont places a cote du CSV d'entree.

In [ ]:
OUTPUT_DIR = DATA_PATH.parent

fenetres.to_csv(OUTPUT_DIR / "resultats_pires_periodes_15j.csv", sep=";", index=False)
resultats_dimensionnement.to_csv(OUTPUT_DIR / "resultats_dimensionnement_optimal.csv", sep=";", index=False)
resultats_dimensionnement_annuel.to_csv(OUTPUT_DIR / "resultats_dimensionnement_optimal_annuel.csv", sep=";", index=False)
simulation_critique.to_csv(OUTPUT_DIR / "simulation_periode_critique_optimum.csv", sep=";", index=False)
simulation_annuelle.to_csv(OUTPUT_DIR / "simulation_annuelle_optimum.csv", sep=";", index=False)
simulation_best_annuel.to_csv(OUTPUT_DIR / "simulation_annuelle_optimum_annuel.csv", sep=";", index=False)

print(f"Exports ecrits dans: {OUTPUT_DIR.resolve()}")